In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Khởi tạo SparkSession
spark = SparkSession.builder \
    .appName("PokemonBigDataAnalytics") \
    .getOrCreate()

print("Spark version:", spark.version)

# Đọc file CSV gốc từ Kaggle
df_pokemon = spark.read.csv(
    "pokemon_complete_stats.csv",
    header=True,
    inferSchema=True
)

# Xem 5 dòng đầu tiên và cấu trúc bảng
df_pokemon.show(5)
df_pokemon.printSchema()

Spark version: 4.0.4
+---+----------+---------+---------+---------------+------+------+---+------+-------+--------------+---------------+-----+--------------------+------------+------------+-----------+-----+---------+------------+
| id|      name|height_dm|weight_hg|base_experience|type_1|type_2| hp|attack|defense|special_attack|special_defense|speed|           abilities|  generation|is_legendary|is_mythical|color|    shape|capture_rate|
+---+----------+---------+---------+---------------+------+------+---+------+-------+--------------+---------------+-----+--------------------+------------+------------+-----------+-----+---------+------------+
|  1| bulbasaur|        7|       69|           64.0| grass|poison| 45|    49|     49|            65|             65|   45|overgrow, chlorop...|generation-i|       false|      false|green|quadruped|          45|
|  2|   ivysaur|       10|      130|          142.0| grass|poison| 60|    62|     63|            80|             80|   60|overgrow, chl

In [ ]:
quality_check = df_pokemon.agg(
    F.count("*").alias("total_rows"),
    F.sum(F.col("name").isNull().cast("int")).alias("missing_name"),
    F.sum(F.col("type_1").isNull().cast("int")).alias("missing_type1"),
    F.sum((F.col("hp") + F.col("attack") + F.col("defense") +
           F.col("special_attack") + F.col("special_defense") + F.col("speed") <= 0).cast("int")).alias("invalid_total_stats")
)

display(quality_check)

df_clean = (
    df_pokemon
    .filter(F.col("name").isNotNull())
    .filter(F.col("type_1").isNotNull())
    .filter(
        (F.col("hp") + F.col("attack") + F.col("defense") +
         F.col("special_attack") + F.col("special_defense") + F.col("speed")) > 0
    )
)

print(f"Số dòng ban đầu: {df_pokemon.count():,}")
print(f"Số dòng sau khi làm sạch: {df_clean.count():,}")

DataFrame[total_rows: bigint, missing_name: bigint, missing_type1: bigint, invalid_total_stats: bigint]

Số dòng ban đầu: 1,351
Số dòng sau khi làm sạch: 1,351


In [ ]:
type_power_ranking = (
    df_clean
    .groupBy("type_1")
    .agg(
        F.count("name").alias("pokemon_count"),
        F.round(
            F.avg(F.col("hp") + F.col("attack") + F.col("defense") +
                  F.col("special_attack") + F.col("special_defense") + F.col("speed")), 2
        ).alias("avg_total_stats"),
        F.round(F.avg("attack"), 2).alias("avg_attack"),
        F.round(F.avg("defense"), 2).alias("avg_defense")
    )
    .orderBy(F.col("avg_total_stats").desc())
)

type_power_ranking.show(10, truncate=False)

+--------+-------------+---------------+----------+-----------+
|type_1  |pokemon_count|avg_total_stats|avg_attack|avg_defense|
+--------+-------------+---------------+----------+-----------+
|dragon  |59           |544.73         |103.24    |83.98      |
|steel   |52           |504.71         |96.46     |113.92     |
|psychic |86           |489.33         |74.87     |72.35      |
|fighting|62           |486.24         |110.44    |82.08      |
|fire    |84           |468.02         |86.19     |70.73      |
|fairy   |35           |463.03         |71.23     |74.43      |
|dark    |63           |462.57         |88.95     |73.46      |
|flying  |13           |457.69         |82.77     |68.38      |
|rock    |83           |456.82         |89.59     |95.6       |
|ground  |50           |449.46         |97.06     |88.2       |
+--------+-------------+---------------+----------+-----------+
only showing top 10 rows


In [ ]:
# Ghi dữ liệu xuống định dạng Parquet
output_path = "pokemon_clean_data.parquet"
df_clean.write.mode("overwrite").parquet(output_path)

print(f"Đã lưu thành công dữ liệu sạch vào định dạng Parquet tại: {output_path}")

Đã lưu thành công dữ liệu sạch vào định dạng Parquet tại: pokemon_clean_data.parquet


In [ ]:
# Chuyển đổi DataFrame PySpark sang Pandas để xuất file CSV gọn nhẹ cho Streamlit
df_clean_pd = df_clean.toPandas()

# Lưu thành file CSV sạch
df_clean_pd.to_csv("pokemon_clean.csv", index=False)

print("Đã xuất thành công file pokemon_clean.csv để làm nguồn dữ liệu cho app!")

Đã xuất thành công file pokemon_clean.csv để làm nguồn dữ liệu cho app!


In [ ]:
import time
import pandas as pd
import duckdb
import polars as pl
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# File path
csv_file = "pokemon_complete_stats.csv"

# 1. Đo thời gian với Pandas
start_time = time.time()
df_pd = pd.read_csv(csv_file)
df_pd_clean = df_pd[df_pd['name'].notnull() & df_pd['type_1'].notnull()]
df_pd_clean['total_calc'] = df_pd_clean[['hp', 'attack', 'defense', 'special_attack', 'special_defense', 'speed']].sum(axis=1)
pd_res = df_pd_clean.groupby('type_1').agg({'name': 'count', 'total_calc': 'mean', 'attack': 'mean', 'defense': 'mean'}).reset_index()
pandas_time = time.time() - start_time

# 2. Đo thời gian với PySpark
spark = SparkSession.builder.appName("Benchmark").getOrCreate()
start_time = time.time()
df_sp = spark.read.csv(csv_file, header=True, inferSchema=True)
df_sp_clean = df_sp.filter(F.col("name").isNotNull() & F.col("type_1").isNotNull())
sp_res = (df_sp_clean.groupBy("type_1")
          .agg(F.count("name").alias("pokemon_count"),
               F.avg(F.col("hp")+F.col("attack")+F.col("defense")+F.col("special_attack")+F.col("special_defense")+F.col("speed")).alias("avg_total"))
          .collect())
pyspark_time = time.time() - start_time

# 3. Đo thời gian với DuckDB (Tối ưu In-memory OLAP)
start_time = time.time()
duck_res = duckdb.query(f"""
    SELECT type_1, COUNT(name) as pokemon_count,
           AVG(hp + attack + defense + special_attack + special_defense + speed) as avg_total
    FROM read_csv_auto('{csv_file}')
    WHERE name IS NOT NULL AND type_1 IS NOT NULL
    GROUP BY type_1
""").df()
duckdb_time = time.time() - start_time

# 4. Đo thời gian với Polars (Vectorized Execution)
start_time = time.time()
df_pl = pl.read_csv(csv_file)
pl_res = (df_pl.filter(pl.col("name").is_not_null() & pl.col("type_1").is_not_null())
          .with_columns((pl.col("hp") + pl.col("attack") + pl.col("defense") + pl.col("special_attack") + pl.col("special_defense") + pl.col("speed")).alias("total_calc"))
          .group_by("type_1")
          .agg([pl.count("name").alias("pokemon_count"), pl.mean("total_calc").alias("avg_total")]))
polars_time = time.time() - start_time

# Summary Benchmark DataFrame
benchmark_df = pd.DataFrame({
    'Tool': ['Polars', 'DuckDB', 'Pandas', 'PySpark (Local Cluster)'],
    'Execution_Time_Seconds': [polars_time, duckdb_time, pandas_time, pyspark_time],
    'Speedup_vs_Pandas': [pandas_time/polars_time, pandas_time/duckdb_time, 1.0, pandas_time/pyspark_time]
})

print("=== KẾT QUẢ BENCHMARK THỜI GIAN XỬ LÝ DỮ LIỆU ===")
print(benchmark_df.to_string(index=False))

=== KẾT QUẢ BENCHMARK THỜI GIAN XỬ LÝ DỮ LIỆU ===
                   Tool  Execution_Time_Seconds  Speedup_vs_Pandas
                 Polars                0.543849           0.156320
                 DuckDB                0.416346           0.204191
                 Pandas                0.085014           1.000000
PySpark (Local Cluster)                9.971231           0.008526


In [10]:
explanation_text = """
================================================================================
GIẢI THÍCH KỸ THUẬT: NGUYÊN LÝ TÍNH TOÁN IN-MEMORY & TỐC ĐỘ CỦA APACHE SPARK
================================================================================
1. Căn cứ khẳng định "Giữ toàn bộ dữ liệu trên RAM":
   - Thay vì liên tục đọc/ghi kết quả trung gian xuống đĩa cứng (HDFS/Disk I/O) như
     kiến trúc Hadoop MapReduce truyền thống, Spark tải tập dữ liệu làm việc (Working set)
     trực tiếp vào bộ nhớ RAM của các nút tính toán (Worker Nodes).
   - Cơ chế RDD (Resilient Distributed Datasets) và Spark In-Memory Storage cho phép
     tái sử dụng dữ liệu trong bộ nhớ qua nhiều vòng lặp mà không tốn chi phí I/O.

2. Căn cứ khẳng định "Tốc độ nhanh hơn 10 - 100 lần":
   - Tốc độ truy xuất dữ liệu từ RAM đạt độ trễ cực thấp (Nanoseconds) so với ổ cứng
     hoặc SSD (Milliseconds - trễ hơn 10,000 - 100,000 lần về mặt vật lý).
   - Tối ưu hóa nhờ Catalyst Optimizer (tạo kế hoạch truy vấn DAG - Directed Acyclic Graph)
     và Bộ máy biên dịch Tungsten Engine (biến đổi bytecode chạy trực tiếp trên JVM).
   - Cơ chế Lazy Evaluation: Spark không thực thi ngay các lệnh Transformation mà chờ
     lệnh Action để tối ưu hóa toàn bộ luồng tính toán (Pipeline Execution), giảm thiểu
     các bước tính thừa.

3. Lưu ý từ Kết quả Thực nghiệm Benchmark:
   - Với tập dữ liệu nhỏ (<1GB) trên 1 máy ảo Colab, các công cụ đơn máy siêu tốc như
     Polars / DuckDB chạy nhanh hơn Pandas & Spark do không bị tốn chi phí khởi tạo
     Spark Context (JVM Overhead & Task Serialization).
   - Tuy nhiên, khi dữ liệu tăng lên quy mô Terabyte hoặc vượt quá RAM đơn máy (Out-of-Core),
     Spark mới phát huy sức mạnh phân tán ngang (Horizontal Scaling) trên cụm nhiều máy chủ.
================================================================================
"""
print(explanation_text)


GIẢI THÍCH KỸ THUẬT: NGUYÊN LÝ TÍNH TOÁN IN-MEMORY & TỐC ĐỘ CỦA APACHE SPARK
1. Căn cứ khẳng định "Giữ toàn bộ dữ liệu trên RAM":
   - Thay vì liên tục đọc/ghi kết quả trung gian xuống đĩa cứng (HDFS/Disk I/O) như
     kiến trúc Hadoop MapReduce truyền thống, Spark tải tập dữ liệu làm việc (Working set)
     trực tiếp vào bộ nhớ RAM của các nút tính toán (Worker Nodes).
   - Cơ chế RDD (Resilient Distributed Datasets) và Spark In-Memory Storage cho phép
     tái sử dụng dữ liệu trong bộ nhớ qua nhiều vòng lặp mà không tốn chi phí I/O.

2. Căn cứ khẳng định "Tốc độ nhanh hơn 10 - 100 lần":
   - Tốc độ truy xuất dữ liệu từ RAM đạt độ trễ cực thấp (Nanoseconds) so với ổ cứng
     hoặc SSD (Milliseconds - trễ hơn 10,000 - 100,000 lần về mặt vật lý).
   - Tối ưu hóa nhờ Catalyst Optimizer (tạo kế hoạch truy vấn DAG - Directed Acyclic Graph)
     và Bộ máy biên dịch Tungsten Engine (biến đổi bytecode chạy trực tiếp trên JVM).
   - Cơ chế Lazy Evaluation: Spark không thực thi ngay các lệnh T

In [15]:
# ==============================================================================
# MỤC 4.2: XÂY DỰNG MÔ HÌNH HỌC MÁY PHÂN TÁN (SPARK MLLIB) DỰ BÁO LEGENDARY
# ==============================================================================
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# 1. Khởi tạo / Kích hoạt lại SparkSession
spark = SparkSession.builder \
    .appName("PokemonSparkMLlib") \
    .getOrCreate()

# 2. Đọc lại file CSV và làm sạch dữ liệu (Tạo df_clean)
csv_file = "pokemon_complete_stats.csv"
df_pokemon = spark.read.csv(csv_file, header=True, inferSchema=True)

df_clean = df_pokemon \
    .filter(F.col("name").isNotNull()) \
    .filter(F.col("type_1").isNotNull()) \
    .filter(
        (F.col("hp") + F.col("attack") + F.col("defense") +
         F.col("special_attack") + F.col("special_defense") + F.col("speed")) > 0
    )

# Ép kiểu cột nhãn 'is_legendary' từ Boolean sang Integer (0 hoặc 1)
df_clean = df_clean.withColumn("is_legendary", F.col("is_legendary").cast("int"))

# 3. Gom nhóm các đặc trưng vào Vector
feature_cols = ['hp', 'attack', 'defense', 'special_attack', 'special_defense', 'speed', 'height_dm', 'weight_hg']
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

df_features = assembler.transform(df_clean)

# 4. Huấn luyện mô hình Random Forest Classifier
rf = RandomForestClassifier(labelCol="is_legendary", featuresCol="features", numTrees=100, seed=42)

# 5. Phân chia tập Train/Test (80% / 20%)
train_df, test_df = df_features.randomSplit([0.8, 0.2], seed=42)

# 6. Huấn luyện và dự báo
model = rf.fit(train_df)
predictions = model.transform(test_df)

# 7. Đánh giá chỉ số ROC-AUC
evaluator = BinaryClassificationEvaluator(
    labelCol="is_legendary",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
auc_score = evaluator.evaluate(predictions)

print("\n=== KẾT QUẢ HUẤN LUYỆN MÔ HÌNH SPARK MLLIB ===")
print(f"Chỉ số ROC-AUC Score đạt được: {auc_score:.4f}")


=== KẾT QUẢ HUẤN LUYỆN MÔ HÌNH SPARK MLLIB ===
Chỉ số ROC-AUC Score đạt được: 0.9024
